[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hudson11111/MMA3001Project/blob/main/notebooks/01_data_audit.ipynb)

# 01 — Data audit: what is in the data, and can we trust the split?

**Work package:** WP1 · **Runtime: CPU is fine** (about 5 minutes) — no GPU quota used.

Questions (PROJECT_PLAN §2, DECISION_LOG D-008):
1. **Coverage** — how big are the defects, where are they, which classes co-occur?
2. **Augmentation accounting** — how many real photographs sit behind the 3,349 training images?
3. **Direct leakage** — does any original photo appear in more than one split?
4. **Temporal leakage** — every image is a numbered video frame. Are neighbouring frames (the
   same tray) spread across train/valid/test? If so, test scores would be optimistic.
5. **Unlabelled images** — do they show good trays, empty conveyor, or missed defects?
6. **A better split** — a time-blocked split with a purge gap, and what it costs.

Learning objectives: data selection, leakage, random vs grouped splits (LO 16).

## 1. Setup

In [ ]:
import os, sys, json, subprocess
REPO_URL = "https://github.com/Hudson11111/MMA3001Project.git"
REPO_DIR = "/content/MMA3001Project"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q main && git -C {REPO_DIR} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -r requirements-colab.txt
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

from pathlib import Path
import itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from porkqc import CLASS_NAMES
from porkqc.config import load_yaml, check_class_names
from porkqc.data.download import download_dataset, fix_data_yaml, read_class_names

cfg = load_yaml("configs/dataset.yaml")
rf = cfg["roboflow"]
data_root = Path(cfg["paths"]["colab_data_dir"])
if not (data_root / "data.yaml").exists():
    data_root = download_dataset(rf["workspace"], rf["project"], rf["version"], data_root, export_format=rf["export_format"])
fix_data_yaml(data_root / "data.yaml")
check_class_names(read_class_names(data_root / "data.yaml"), CLASS_NAMES)
SPLITS = ("train", "valid", "test")
FIG = Path("results/figures"); FIG.mkdir(parents=True, exist_ok=True)
AUD = Path("results/audit"); AUD.mkdir(parents=True, exist_ok=True)
git_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("data:", data_root, "| commit:", git_commit)

## 2. Split overview (recap of notebook 00)

In [ ]:
from porkqc.data.audit import summarise_dataset
summ = summarise_dataset(data_root, len(CLASS_NAMES))
overview = pd.DataFrame([{ "split": k, "images": s.n_images, "unique originals": s.n_unique_originals,
    "null images": s.n_null_images, "objects": s.n_instances,
    "missing/invalid labels": s.n_missing_labels + s.n_invalid_label_files} for k, s in summ.items()]).set_index("split")
per_class = pd.DataFrame({k: s.instances_per_class for k, s in summ.items()}, index=CLASS_NAMES)
display(overview); display(per_class)

## 3. Box statistics per class

Sizes are fractions of the image area. "Small" uses the COCO definition (< 32 × 32 px,
i.e. < 0.26% of a 720 × 540 image) — small objects are typically hardest to detect.

In [ ]:
from porkqc.data.audit import box_table
boxes = pd.DataFrame([r for s in SPLITS for r in box_table(data_root / s, len(CLASS_NAMES), split=s)])
boxes["class"] = boxes["class_id"].map(dict(enumerate(CLASS_NAMES)))
SMALL = 32 * 32 / (720 * 540)
stats = boxes.groupby("class").agg(
    boxes=("area_frac", "size"),
    median_area_pct=("area_frac", lambda a: 100 * a.median()),
    q10_area_pct=("area_frac", lambda a: 100 * a.quantile(0.1)),
    q90_area_pct=("area_frac", lambda a: 100 * a.quantile(0.9)),
    small_pct=("area_frac", lambda a: 100 * (a < SMALL).mean()),
    median_aspect_wh=("aspect", "median"),
).round(2)
display(stats)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
data = [boxes.loc[boxes["class"] == c, "area_frac"] * 100 for c in CLASS_NAMES]
axes[0].boxplot(data, vert=False); axes[0].set_yticks(range(1, len(CLASS_NAMES) + 1), CLASS_NAMES); axes[0].set_xscale("log")
axes[0].axvline(SMALL * 100, ls="--", c="grey"); axes[0].set_xlabel("box area (% of image, log scale)")
axes[0].set_title("Defect size by class (dashed: COCO 'small')")
for k, c in enumerate(CLASS_NAMES):
    sub = boxes[boxes["class"] == c]
    axes[1].scatter(sub["cx"], sub["cy"], s=4, alpha=0.4, label=c)
axes[1].invert_yaxis(); axes[1].set_xlim(0, 1); axes[1].set_xlabel("x (normalised)"); axes[1].set_ylabel("y")
axes[1].set_title("Box centres (image coordinates)"); axes[1].legend(markerscale=4, fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "01_box_statistics.png", dpi=110); plt.show()

## 4. Defects per image and class co-occurrence

Counted on **unique originals** of the training split (augmented copies would triple-count).

In [ ]:
from porkqc.data.audit import image_classes, original_stem
rows = []
for s in SPLITS:
    for f, cls in image_classes(data_root / s, len(CLASS_NAMES)).items():
        rows.append({"split": s, "file": f, "original": original_stem(f), "classes": cls})
imgs = pd.DataFrame(rows)
train_orig = imgs[imgs.split == "train"].drop_duplicates("original")

co = np.zeros((len(CLASS_NAMES),) * 2, dtype=int)
for cls in train_orig["classes"]:
    for a in cls:
        for b in cls:
            co[a, b] += 1
co_df = pd.DataFrame(co, index=CLASS_NAMES, columns=CLASS_NAMES)
print("Images (train originals) containing class i (row) AND class j (column); diagonal = images with class i")
display(co_df)

n_cls = train_orig["classes"].map(len)
print("Distinct defect classes per training original:", n_cls.value_counts().sort_index().to_dict())
bpi = boxes[boxes.split == "train"].drop_duplicates(["original", "cx", "cy", "class_id"]).groupby("original").size()
print("Boxes per training original: median", bpi.median(), "| max", bpi.max())

## 5. What is one "original image"? Augmented copies vs same-named frames

Roboflow names every exported file `<original>_jpg.rf.<hash>.jpg`; augmented copies of one image
share `<original>`. In the development sample, however, two **different** trays were both named
`frame_with_red_992` — apparently frames from different videos numbered the same way. So the
name alone does not identify an image.

**Method:** compare files that share a name with a *standardised greyscale thumbnail*
(64 × 48, zero mean, unit variance) and their correlation. Calibration:
* **positive control (manufactured):** apply Roboflow-range colour augmentation (saturation ±29 %,
  exposure ±10 %) to real images — these *are* copies and must score ≈ 1;
* **negative control:** pairs of images with different names — different frames.

Files correlating above the threshold are copies of one **scene**; the rest become separate scenes.

In [ ]:
from PIL import Image, ImageEnhance
from porkqc.data.duplicates import grey_thumbnail, thumbnail_correlation
from porkqc.data.resplit import collect_frames, scene_of

rng_c = np.random.default_rng(0)
sample = imgs.sample(min(60, len(imgs)), random_state=0)
pos = []
for r in sample.itertuples():
    with Image.open(data_root / r.split / "images" / r.file) as im:
        im = im.convert("RGB")
        aug = ImageEnhance.Brightness(ImageEnhance.Color(im).enhance(rng_c.uniform(0.71, 1.29))).enhance(rng_c.uniform(0.9, 1.1))
        pos.append(thumbnail_correlation(grey_thumbnail(im), grey_thumbnail(aug)))
thumbs = {r.file: grey_thumbnail(data_root / r.split / "images" / r.file) for r in sample.itertuples()}
names = list(thumbs)
neg = [thumbnail_correlation(thumbs[a], thumbs[b]) for a, b in itertools.combinations(names, 2)
       if original_stem(a) != original_stem(b)]
COPY_THRESHOLD = 0.99
print(f"positive control (synthetic copies): min {min(pos):.4f}, median {np.median(pos):.4f}")
print(f"negative control (different names):  max {max(neg):.3f}, median {np.median(neg):.3f}")
print(f"threshold {COPY_THRESHOLD}: separates the controls" if min(pos) > COPY_THRESHOLD > max(neg)
      else "WARNING: controls overlap at this threshold - tell Claude")

frames_map, files_map, unparsed = collect_frames(data_root, copy_threshold=COPY_THRESHOLD)
scene_by_file = scene_of(files_map)
imgs["scene"] = imgs.file.map(scene_by_file)
n_names = imgs.groupby("split")["original"].nunique()
n_scenes = imgs.groupby("split")["scene"].nunique()
copies = imgs[imgs.split == "train"].groupby("scene").size()
print("\nimages / distinct names / distinct scenes per split:")
display(pd.DataFrame({"images": imgs.groupby("split").size(), "names": n_names, "scenes": n_scenes}))
print("Training copies per scene:", copies.value_counts().sort_index().to_dict())
multi = {s for s in files_map if "#" in s}
print(f"names shared by different images: {len({s.split('#')[0] for s in multi})} "
      f"(creating {len(multi)} extra scenes)")
scene_splits = imgs.groupby("scene")["split"].agg(lambda x: tuple(sorted(set(x))))
shared = {"+".join(k): list(v.index[:5]) for k, v in scene_splits[scene_splits.map(len) > 1].groupby(scene_splits)}
print("DIRECT leakage - the same image (copies) in several splits:", {k: len(v) for k, v in shared.items()} or "none")

## 6. Temporal structure — every image is a video frame

File names look like `frame_with_red_2247_jpg…`: the number is the frame index in the
conveyor video. Neighbouring frames usually show **the same tray**. If Roboflow put
neighbouring frames into different splits, the model is tested on trays it has already
seen during training — **leakage**, which makes test scores optimistic.

In [ ]:
from porkqc.data.resplit import nearest_other_gap
print(f"{len(frames_map)} scenes have a frame number; {len(unparsed)} names could not be parsed", unparsed[:5])
orig_split = {sc: fs[0].parent.parent.name for sc, fs in files_map.items()}  # split of each scene
fr = pd.DataFrame({"original": list(frames_map), "frame": list(frames_map.values())})
fr["split"] = fr["original"].map(orig_split)  # 'original' = scene id
fr = fr.sort_values("frame").reset_index(drop=True)
print(f"frames {fr.frame.min()}-{fr.frame.max()}; median spacing between consecutive dataset frames: "
      f"{fr.frame.diff().median():.0f}")

fig, ax = plt.subplots(figsize=(12, 2.4))
for k, s in enumerate(SPLITS):
    sub = fr[fr.split == s]
    ax.scatter(sub.frame, np.full(len(sub), k), s=40, marker="|")
ax.set_yticks(range(3), SPLITS); ax.set_xlabel("frame number"); ax.set_ylim(-0.6, 2.6)
ax.set_title("Roboflow's split along the video: the three splits are interleaved frame by frame")
fig.tight_layout(); fig.savefig(FIG / "01_split_timeline.png", dpi=110); plt.show()

GAPS = (1, 2, 5, 10, 25, 50, 100)
def gap_table(frames, splits):
    rows = []
    for s, others in (("valid", ["train"]), ("test", ["train"]), ("test", ["valid"])):
        g = nearest_other_gap(frames, splits, s, others)
        rows.append({"split": s, "nearest in": "+".join(others), "median gap": float(np.median(g)),
                     **{f"<= {k}": f"{100 * (g <= k).mean():.0f}%" for k in GAPS}})
    return pd.DataFrame(rows)
leak_roboflow = gap_table(fr.frame.values, fr.split.values)
print("Share of VALID/TEST frames with a frame from another split within k frames:")
display(leak_roboflow)

## 7. How many frames does one tray stay in view?

This sets how far apart two splits must be. **Criterion:** two frames show the same tray if
the tray outline and meat arrangement match (allowing for movement along the conveyor).
Each row below is a run of neighbouring frames from the whole dataset (all splits) around a
VALID/TEST frame. Note the largest frame gap over which the *same* tray is still visible.

In [ ]:
from porkqc.data.labels import read_label_file, yolo_to_xyxy
import matplotlib.patches as patches
from PIL import Image

def show_frame(ax, original, title_extra=""):
    p = files_map[original][0]
    with Image.open(p) as im:
        ax.imshow(im.convert("RGB")); w, h = im.size
    lab = read_label_file(p.parent.parent / "labels" / f"{p.stem}.txt", len(CLASS_NAMES))
    for (c, *_), (x1, y1, x2, y2) in zip(lab, yolo_to_xyxy(lab, w, h)):
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="yellow", lw=1.5))
        ax.text(x1, y1 - 3, CLASS_NAMES[int(c)], color="yellow", fontsize=6, backgroundcolor="black")
    ax.set_title(f"{frames_map[original]} ({orig_split[original]}){title_extra}", fontsize=8); ax.axis("off")

rng_anchor = np.random.default_rng(1)
anchors = rng_anchor.choice(fr[fr.split != "train"].index, 4, replace=False)
N_SHOW = 8
fig, axes = plt.subplots(len(anchors), N_SHOW, figsize=(2.6 * N_SHOW, 2.2 * len(anchors)))
for row, a in zip(axes, anchors):
    lo = max(0, a - N_SHOW // 2); run = fr.iloc[lo:lo + N_SHOW]
    for ax, (_, r) in zip(row, run.iterrows()):
        show_frame(ax, r.original)
fig.suptitle("Runs of neighbouring frames (frame number, split). Same tray across frames => leakage if splits differ", fontsize=10)
fig.tight_layout(); fig.savefig(FIG / "01_frame_runs.png", dpi=80); plt.show()

## 8. Images without labels — good trays or missing labels?

In the development sample, frame 2247 (VALID) is labelled *unsealed*, while frame 2248 (TEST)
shows the same tray **with no label at all**. So "no label" may mean *no defect*, *no tray in
view*, or *a defect the annotator missed*. This decides whether a PASS decision can be
validated and how much to trust false-positive counts.

**Manual review (please do this — it is your validation evidence).** For each unlabelled
VALID/TEST image below, assign one code:

| Code | Meaning |
|---|---|
| `E` | empty conveyor, or tray mostly out of view — nothing to inspect |
| `G` | tray clearly visible and acceptable — a genuine PASS |
| `M` | a defect is visible but not labelled — missing label |
| `U` | unsure |

The titles show the frame number and the labels of the nearest labelled frame (± 5 frames),
which is a hint, not the answer. Fill the `review` column of `results/audit/null_image_review.csv`
(or just tell Claude the counts per code).

In [ ]:
lab_by_orig = {}
for o in fr.original:
    p = files_map[o][0]
    lab = read_label_file(p.parent.parent / "labels" / f"{p.stem}.txt", len(CLASS_NAMES))
    lab_by_orig[o] = sorted({CLASS_NAMES[int(c)] for c in lab[:, 0]})
fr["labels"] = fr.original.map(lab_by_orig)
fr["is_null"] = fr.labels.map(len) == 0

nulls = fr[fr.is_null & fr.split.isin(["valid", "test"])].copy()
def nearest_labelled(f):
    near = fr[(~fr.is_null) & ((fr.frame - f).abs() <= 5)]
    return "; ".join(f"{r.frame}:{'+'.join(r.labels)}" for r in near.itertuples()) or "-"
nulls["nearest_labelled_within_5"] = nulls.frame.map(nearest_labelled)
print(f"Unlabelled images: train {int((fr.is_null & (fr.split == 'train')).sum())}, "
      f"valid {int((nulls.split == 'valid').sum())}, test {int((nulls.split == 'test').sum())}")

cols = 4
rows = int(np.ceil(len(nulls) / cols)) or 1
fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.7 * rows))
for ax in np.ravel(axes): ax.axis("off")
for ax, r in zip(np.ravel(axes), nulls.itertuples()):
    show_frame(ax, r.original, f"\nnear: {r.nearest_labelled_within_5[:40]}")
fig.tight_layout(); fig.savefig(FIG / "01_null_images_review.png", dpi=80); plt.show()

nulls.assign(review="")[["original", "split", "frame", "nearest_labelled_within_5", "review"]] \
    .to_csv(AUD / "null_image_review.csv", index=False)
print("Review sheet:", AUD / "null_image_review.csv")

## 9. A time-blocked split that removes the leakage

Frames are grouped into contiguous **blocks**; whole blocks are assigned to train/valid/test,
and frames within a **gap** of a different split's frame are dropped (from TRAIN first, so the
evaluation sets keep their size). This is the time-series analogue of a grouped split (LO 16).

The table shows the trade-off: a larger gap removes more leakage risk but discards more
training frames. Settings used for training come from `configs/dataset.yaml → blocked_split`
(provisional until sections 7–8 have been reviewed).

In [ ]:
from porkqc.data.resplit import assign_time_blocks
bcfg = cfg["blocked_split"]

def evaluate_assignment(assign):
    s = pd.Series(assign)
    kept = s.dropna()
    f = np.array([frames_map[o] for o in kept.index]); sp = kept.values.astype(str)
    out = {k: int((s == k).sum()) for k in SPLITS}
    out["purged"] = int(s.isna().sum())
    out["min cross-split gap"] = int(min(nearest_other_gap(f, sp, t, [o for o in SPLITS if o != t]).min()
                                         for t in ("valid", "test")))
    return out

sens = pd.DataFrame([{"block_size": b, "gap": g, **evaluate_assignment(
    assign_time_blocks(frames_map, b, bcfg["fractions"], g, bcfg["seed"]))}
    for b in (150, 300, 600) for g in (0, 10, 25, 50, 100)])
print("Originals per split for different block sizes and gaps (Roboflow: train "
      f"{int((fr.split == 'train').sum())}, valid {int((fr.split == 'valid').sum())}, test {int((fr.split == 'test').sum())})")
display(sens)

assign = assign_time_blocks(frames_map, bcfg["block_size"], bcfg["fractions"], bcfg["gap"], bcfg["seed"])
fr["blocked_split"] = fr.original.map(assign)
kept = fr.dropna(subset=["blocked_split"])
leak_blocked = gap_table(kept.frame.values, kept.blocked_split.values.astype(str))
print(f"Chosen: block_size={bcfg['block_size']}, gap={bcfg['gap']}. Leakage check on the new split:")
display(leak_blocked)

boxes["scene"] = boxes.file.map(scene_by_file)
inst = boxes.drop_duplicates(["scene", "cx", "cy", "class_id"]).copy()
inst["blocked_split"] = inst.scene.map(assign)
cls_counts = inst.dropna(subset=["blocked_split"]).pivot_table(index="class", columns="blocked_split",
                                                                values="cx", aggfunc="size", fill_value=0)
print("Objects per class in the blocked split (one copy per original):"); display(cls_counts)
print("Unlabelled originals per blocked split:", kept[kept.is_null].blocked_split.value_counts().to_dict())
fr[["original", "frame", "split", "blocked_split"]].to_csv(AUD / "blocked_split_assignment.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 2.4))
for k, s in enumerate(SPLITS):
    sub = kept[kept.blocked_split == s]
    ax.scatter(sub.frame, np.full(len(sub), k), s=40, marker="|")
ax.set_yticks(range(3), SPLITS); ax.set_xlabel("frame number"); ax.set_ylim(-0.6, 2.6)
ax.set_title(f"Time-blocked split (block {bcfg['block_size']} frames, gap {bcfg['gap']})")
fig.tight_layout(); fig.savefig(FIG / "01_blocked_split_timeline.png", dpi=110); plt.show()

## 10. Appendix — why perceptual hashing was *not* used (an unsuccessful method)

The first plan was to find near-duplicates by image similarity (64-bit difference hash,
`porkqc.data.duplicates`). The controls show why it fails here: the dark conveyor background
dominates every image, so **different** frames look alike, while **neighbouring** frames of the
same tray differ because the tray has moved. A method is only as good as its controls.

In [ ]:
from porkqc.data.duplicates import dhash, hash_images, paired_hamming
first_copy = [files_map[o][0] for o in fr.original]
H = hash_images(first_copy)
fnum = fr.frame.values
rng_h = np.random.default_rng(0)
i = rng_h.integers(0, len(fr), 20000); j = rng_h.integers(0, len(fr), 20000)
ok = i != j
d_rand = paired_hamming(H[i[ok]], H[j[ok]]); g_rand = np.abs(fnum[i[ok]] - fnum[j[ok]])
d_next = paired_hamming(H[:-1], H[1:]); g_next = np.diff(fnum)   # consecutive dataset frames
bins = [0, 2, 5, 10, 25, 100, 1000, 100000]
tab = pd.DataFrame({"gap": np.concatenate([g_rand, g_next]), "dist": np.concatenate([d_rand, d_next])})
tab["frame gap"] = pd.cut(tab.gap, bins, right=False)
print("dHash distance (bits) between frames, by how far apart they are in the video:")
display(tab.groupby("frame gap", observed=True)["dist"].agg(["count", "median", "min", "max"]))

d_aug = []
for p in first_copy[:60]:   # manufactured positive control: colour-augmented copies
    with Image.open(p) as im:
        im = im.convert("RGB")
        aug = ImageEnhance.Brightness(ImageEnhance.Color(im).enhance(rng_h.uniform(0.71, 1.29))).enhance(rng_h.uniform(0.9, 1.1))
        d_aug.append(paired_hamming(dhash(im)[None], dhash(aug)[None])[0])
print(f"colour-augmented copies: median {np.median(d_aug):.0f} bits, max {max(d_aug)}; "
      f"random different frames: median {np.median(d_rand):.0f}, min {d_rand.min()} "
      "(unrelated natural images would be ~32). The two overlap, so a hash threshold cannot separate "
      "'same image' from 'different frame' on this dataset.")

## 11. Save the audit

In [ ]:
import shutil
audit = {
    "git_commit": git_commit,
    "overview": overview.reset_index().to_dict(orient="records"),
    "objects_per_class": per_class.to_dict(),
    "box_stats": stats.reset_index().to_dict(orient="records"),
    "cooccurrence_train_originals": co_df.to_dict(),
    "copy_detection": {"threshold": COPY_THRESHOLD, "positive_min": float(min(pos)), "negative_max": float(max(neg))},
    "scenes_per_split": n_scenes.to_dict(), "names_per_split": n_names.to_dict(),
    "train_copies_per_scene": {int(k): int(v) for k, v in copies.value_counts().items()},
    "names_shared_by_different_images": len({s.split('#')[0] for s in multi}),
    "direct_leakage": {k: len(v) for k, v in shared.items()},
    "frames": {"n": int(len(fr)), "unparsed": len(unparsed), "min": int(fr.frame.min()), "max": int(fr.frame.max()),
               "median_spacing": float(fr.frame.diff().median())},
    "leakage_roboflow_split": leak_roboflow.to_dict(orient="records"),
    "unlabelled_originals": {s: int((fr.is_null & (fr.split == s)).sum()) for s in SPLITS},
    "blocked_split": {"settings": bcfg, "sensitivity": sens.to_dict(orient="records"),
                      "leakage": leak_blocked.to_dict(orient="records"),
                      "class_counts": cls_counts.to_dict()},
}
with open(AUD / "audit_summary.json", "w") as fh:
    json.dump(audit, fh, indent=1, default=str)
print(json.dumps(audit, indent=1, default=str)[:2500])

from porkqc.env import get_secret
from porkqc.colab_utils import push_results
paths = [AUD] + sorted(FIG.glob("01_*.png"))
try:
    sha = push_results(REPO_DIR, [str(p) for p in paths], "results: data audit (Colab)", get_secret("GITHUB_TOKEN"))
    print("Pushed to 'colab-results':", sha[:7])
except (RuntimeError, ValueError) as exc:
    print("Not pushed:", exc)
    archive = shutil.make_archive("/content/results_audit", "zip", ".", "results")
    from google.colab import files as colab_files
    colab_files.download(archive)

## 12. Hand back
1. **Review the unlabelled images** (section 8): fill the `review` column in
   `results/audit/null_image_review.csv` (Files panel → double-click to open, or download), or just
   tell Claude how many were `E`, `G`, `M`, `U`.
2. From section 7: roughly how many frames does the same tray stay visible?
3. Tell Claude "audit pushed" (or attach the zip), and optionally *File → Save a copy in GitHub*
   (path `notebooks/01_data_audit.ipynb`).